# Task 3

Due to the simplicity of KNN for Classification, let's focus on using a Pipeline and a GridSearchCV tool, since these skills can be generalized for any model.


## The Sonar Data

### Detecting a Rock or a Mine

Sonar (sound navigation ranging) is a technique that uses sound propagation (usually underwater, as in submarine navigation) to navigate, communicate with or detect objects on or under the surface of the water, such as other vessels.



The data set contains the response metrics for 60 separate sonar frequencies sent out against a known mine field (and known rocks). These frequencies are then labeled with the known object they were beaming the sound at (either a rock or a mine).



Our main goal is to create a machine learning model capable of detecting the difference between a rock or a mine based on the response of the 60 separate sonar frequencies.


Data Source: https://archive.ics.uci.edu/ml/datasets/Connectionist+Bench+(Sonar,+Mines+vs.+Rocks)

### Complete the Tasks in bold

**TASK: Run the cells below to load the data.**

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('sonar.all-data.csv')

In [3]:
df.head()
df.shape

(208, 61)

## Train | Test Split

Our approach here will be one of using Cross Validation on 90% of the dataset, and then judging our results on a final test set of 10% to evaluate our model.

**TASK: Split the data into features and labels, and then split into a training set and test set, with 90% for Cross-Validation training, and 10% for a final test set.**

*Note: The solution uses a random_state=42*

In [4]:
from sklearn.model_selection import train_test_split

In [5]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42
)
print(X_train.shape)
print(X_test.shape)

(187, 60)
(21, 60)


**TASK: Create a Pipeline that contains both a StandardScaler and a KNN model**

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

In [7]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# This pipeline combines two steps into a single workflow.
#  First, the StandardScaler standardizes the input features so that each feature has a mean of 0 and a standard deviation of 1. 
#  This step is important because the KNN algorithm is distance-based, and features with larger scales could otherwise dominate the distance calculation. 
#  After scaling, the KNeighborsClassifier is applied to perform the classification task.
#  Using a pipeline ensures that preprocessing and model training are applied consistently during training, cross-validation, and testing.

**TASK: Perform a grid-search with the pipeline to test various values of k and report back the best performing parameters.**

In [8]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid = {
    'knn__n_neighbors': list(range(1, 31))
}

# Create the GridSearchCV object
grid = GridSearchCV(pipe, param_grid, cv=10)

# Fit the grid search to the training data
grid.fit(X_train, y_train)

# Print the best parameters
print(grid.best_params_)
print("Best Cross Validation Score:", grid.best_score_)

# In this step, GridSearchCV is used to find the best value of k (number of neighbors) for the KNN model. 
# A parameter grid is defined to test different values of n_neighbors from 1 to 30. 
# Because the KNN model is inside the pipeline, the parameter is referenced as knn__n_neighbors. 
# The grid search evaluates each value using 10-fold cross-validation on the training dataset and selects the parameter combination that produces the best performance. 
# After fitting the grid search, grid.best_params_ returns the value of k that achieved the highest cross-validation score.


{'knn__n_neighbors': 1}
Best Cross Validation Score: 0.861111111111111


### Final Model Evaluation

**TASK: Using the grid classifier object from the previous step, get a final performance classification report and confusion matrix.**

In [9]:
from sklearn.metrics import classification_report, confusion_matrix

# Make predictions on the test set
y_pred = grid.predict(X_test)

# Print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Print classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# In this step, the optimized model obtained from GridSearchCV is evaluated on the test dataset that was not used during training or cross-validation.
#  First, predictions are generated using grid.predict(X_test). 
#  Then, a confusion matrix is printed to show how many samples were correctly or incorrectly classified as rocks or mines.
#   Finally, a classification report is displayed, which includes important evaluation metrics such as precision, recall, f1-score, and accuracy.
#  These metrics provide a detailed understanding of the model's performance on the unseen test data.

Confusion Matrix:
[[12  1]
 [ 1  7]]

Classification Report:
              precision    recall  f1-score   support

           M       0.92      0.92      0.92        13
           R       0.88      0.88      0.88         8

    accuracy                           0.90        21
   macro avg       0.90      0.90      0.90        21
weighted avg       0.90      0.90      0.90        21

